<a href="https://colab.research.google.com/github/WashingtonBM/Dashboard-da-Porsche/blob/main/Dashboard_Porsch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# sanitize_porsche.py


"""
Agente de Higienização de Vendas Porsche
=======================================

Lê um arquivo XLSX contendo registros de vendas da Porsche e gera um novo
arquivo Excel contendo as colunas originais e suas respectivas colunas
higienizadas imediatamente após cada coluna de origem.

Uso:
    python sanitize_porsche.py entrada.xlsx saida.xlsx

Ou:

    python sanitize_porsche.py

Neste caso utilizará:
    porsche_database.xlsx
    porsche_database_sanitized.xlsx
"""

from __future__ import annotations

import argparse
import datetime as dt
import re
import unicodedata
from pathlib import Path

from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill


INPUT_PATH = Path("porsche_database.xlsx")
OUTPUT_PATH = Path("porsche_database_sanitized.xlsx")

INVALID = "INVALID"

# ============================================================
# DATAS
# ============================================================

MONTHS = {
    "jan": 1,
    "january": 1,
    "feb": 2,
    "february": 2,
    "mar": 3,
    "march": 3,
    "apr": 4,
    "april": 4,
    "may": 5,
    "jun": 6,
    "june": 6,
    "jul": 7,
    "july": 7,
    "aug": 8,
    "august": 8,
    "sep": 9,
    "sept": 9,
    "september": 9,
    "oct": 10,
    "october": 10,
    "nov": 11,
    "november": 11,
    "dec": 12,
    "december": 12,
}


def safe_date(y, m, d):
    try:
        return dt.date(y, m, d).strftime("%Y-%m-%d")
    except Exception:
        return INVALID


def sanitize_date(value):
    if value is None:
        return INVALID

    if isinstance(value, (dt.date, dt.datetime)):
        return value.strftime("%Y-%m-%d")

    s = str(value).strip()

    if not s:
        return INVALID

    m = re.match(r"^(\d{4})[-/.](\d{1,2})[-/.](\d{1,2})$", s)
    if m:
        return safe_date(
            int(m.group(1)),
            int(m.group(2)),
            int(m.group(3))
        )

    m = re.match(r"^(\d{1,2})[-/](\d{1,2})[-/](\d{4})$", s)
    if m:
        return safe_date(
            int(m.group(3)),
            int(m.group(1)),
            int(m.group(2))
        )

    m = re.match(r"^(\d{1,2})[-/](\d{1,2})[-/](\d{2})$", s)
    if m:
        year = 2000 + int(m.group(3))
        return safe_date(
            year,
            int(m.group(1)),
            int(m.group(2))
        )

    m = re.match(
        r"^([A-Za-z]+)\s+(\d{1,2})(?:st|nd|rd|th)?\,?s+(\d{4})$",
        s
    )

    if m:
        mon = m.group(1).lower()
        if mon in MONTHS:
            return safe_date(
                int(m.group(3)),
                MONTHS[mon],
                int(m.group(2))
            )

    return INVALID


# ============================================================
# MODELOS PORSCHE
# ============================================================

CANONICAL_MODELS = [
    "911 Carrera",
    "911 Carrera S",
    "911 Carrera GTS",
    "911 Turbo",
    "911 Turbo S",
    "911 GT3",
    "911 GT3 RS",
    "911 Dakar",
    "911 Targa 4",
    "911 Targa 4S",
    "718 Cayman",
    "718 Cayman S",
    "718 Cayman GT4 RS",
    "718 Boxster",
    "718 Boxster GTS",
    "718 Spyder RS",
    "Cayenne",
    "Cayenne S",
    "Cayenne Coupe",
    "Cayenne E-Hybrid",
    "Cayenne Turbo",
    "Cayenne Turbo GT",
    "Macan",
    "Macan S",
    "Macan T",
    "Macan GTS",
    "Macan Electric",
    "Panamera",
    "Panamera 4",
    "Panamera 4S",
    "Panamera Turbo",
    "Panamera Turbo S",
    "Panamera 4 E-Hybrid",
    "Taycan",
    "Taycan 4S",
    "Taycan GTS",
    "Taycan Turbo",
    "Taycan Turbo S",
    "Taycan Cross Turismo",
]

MODEL_LOOKUP = {
    m.lower(): m
    for m in CANONICAL_MODELS
}


def sanitize_model(value):

    if value is None:
        return INVALID

    text = " ".join(str(value).split()).strip()

    if not text:
        return INVALID

    key = text.lower()

    if key in MODEL_LOOKUP:
        return MODEL_LOOKUP[key]

    return text.title()


# ============================================================
# ANO
# ============================================================

def sanitize_year(value):

    if value is None:
        return INVALID

    s = str(value).strip()

    m = re.search(r"(\d{4})", s)

    if m:
        year = int(m.group(1))

        if 1990 <= year <= 2035:
            return str(year)

    m = re.match(r"(\d{2})[\s\-\.](\d{2})", s)

    if m:
        year = int(m.group(1) + m.group(2))

        if 1990 <= year <= 2035:
            return str(year)

    return INVALID


# ============================================================
# PREÇO
# ============================================================

def parse_numeric_price(text):

    text = text.strip()

    if "." in text and "," in text:

        if text.rfind(",") > text.rfind("."):
            text = text.replace(".", "")
            text = text.replace(",", ".")

        else:
            text = text.replace(",", "")

    elif "," in text:

        if len(text.split(",")[-1]) == 2:
            text = text.replace(",", ".")
        else:
            text = text.replace(",", "")

    elif "." in text:

        if len(text.split(".")[-1]) != 2:
            text = text.replace(".", "")

    try:
        return float(text)
    except:
        return None


def sanitize_price(value):

    if value is None:
        return INVALID

    s = str(value).lower()

    s = s.replace("$", "")
    s = s.replace("usd", "")

    multiplier = 1

    if "mil" in s:
        multiplier = 1000

    match = re.search(r"[\d.,]+", s)

    if not match:
        return INVALID

    n = parse_numeric_price(match.group())

    if n is None:
        return INVALID

    return f"{n * multiplier:.2f}"


# ============================================================
# MILEAGE
# ============================================================

def sanitize_mileage(value):

    if value is None:
        return INVALID

    s = str(value).lower()

    if any(x in s for x in [
        "new",
        "carro novo",
        "novo",
        "zero miles",
        "zero milhas"
    ]):
        return "0"

    match = re.search(r"[\d.,]+", s)

    if not match:
        return INVALID

    n = parse_numeric_price(match.group())

    if n is None:
        return INVALID

    if "km" in s:
        n *= 0.621371

    return str(round(n))


# ============================================================
# PAGAMENTO
# ============================================================

PAYMENTS = {
    "credit card": "Cartão de Crédito",
    "debit card": "Cartão de Débito",
    "bank transfer": "Transferência Bancária",
    "wire": "Transferência Bancária",
    "financing": "Financiamento",
    "lease": "Arrendamento",
    "cash": "Dinheiro",
    "ach": "Pagamento ACH",
    "crypto": "Pagamento em Criptomoedas",
}


def sanitize_payment(value):

    if value is None:
        return INVALID

    key = str(value).lower().strip()

    for k, v in PAYMENTS.items():
        if k in key:
            return v

    return str(value).title()


# ============================================================
# CIDADE
# ============================================================

def sanitize_city(value):

    if value is None:
        return INVALID

    return str(value).title().strip()


# ============================================================
# ESTADOS
# ============================================================

US_STATES = {
    "california": "CA",
    "new york": "NY",
    "texas": "TX",
    "florida": "FL",
    "illinois": "IL",
    "nevada": "NV",
    "washington": "WA",
    "arizona": "AZ",
    "georgia": "GA",
}


def sanitize_state(value):

    if value is None:
        return INVALID

    text = str(value).strip()

    if len(text) == 2:
        return text.upper()

    key = text.lower()

    return US_STATES.get(key, INVALID)


# ============================================================
# STATUS
# ============================================================

STATUS_MAP = {
    "delivered": "Entregue",
    "deliverd": "Entregue",
    "pending": "Pendente",
    "in transit": "Em trânsito",
    "cancelled": "Cancelado",
    "canceled": "Cancelado",
    "awaiting delivery": "Aguardando Entrega",
    "awaiting pickup": "Aguardando Recolhimento",
    "awaiting approval": "Aguardando Aprovação",
    "under review": "Em análise",
    "shipped": "Enviado",
}


def sanitize_delivery(value):

    if value is None:
        return INVALID

    key = str(value).lower().strip()

    key = re.sub(r"[^\w\s]", "", key)

    return STATUS_MAP.get(
        key,
        str(value).title()
    )


# ============================================================
# PIPELINE
# ============================================================

PIPELINE = [
    ("sale_id", None, None),
    ("sale_date", "SaleDateSanitized", sanitize_date),
    ("customer_name", None, None),
    ("porsche_model", "PorscheModelSanitized", sanitize_model),
    ("model_year", "ModelYearSanitized", sanitize_year),
    ("sale_price", "SalesPriceSanitized", sanitize_price),
    ("vehicle_mileage", "VehicleMileageSanitized", sanitize_mileage),
    ("payment_method", "PayMethodSanitized", sanitize_payment),
    ("city", "CitySanitized", sanitize_city),
    ("state", "StateSanitized", sanitize_state),
    ("salesperson", None, None),
    ("delivery_status", "DeliveryStatusSanitized", sanitize_delivery),
]


# ============================================================
# PROCESSAMENTO
# ============================================================

def process_workbook(input_file, output_file):
    input_path = Path(input_file)

    if not input_path.exists():
        print(f"Erro: O arquivo de entrada '{input_file}' não foi encontrado. Por favor, carregue o arquivo XLSX correto.")
        return

    if input_path.suffix.lower() != '.xlsx':
        print(f"Erro: O arquivo de entrada '{input_file}' não é um arquivo XLSX válido. Por favor, forneça um arquivo com extensão .xlsx.")
        return

    wb = load_workbook(input_file)
    ws = wb.active

    rows = list(ws.iter_rows(values_only=True))

    header = [str(x) for x in rows[0]]

    name_to_idx = {
        name: idx
        for idx, name in enumerate(header)
    }

    out_wb = Workbook()
    out_ws = out_wb.active

    out_header = []

    for src, san, _ in PIPELINE:

        out_header.append(src)

        if san:
            out_header.append(san)

    out_ws.append(out_header)

    for row in rows[1:]:  # Skip header row

        output_row = []

        for src, san, fn in PIPELINE:
            # Ensure the column exists before trying to access it
            if src in name_to_idx:
                raw = row[name_to_idx[src]]
            else:
                raw = None # Or handle as an error, depending on requirements

            output_row.append(raw)

            if san:
                output_row.append(fn(raw))

        out_ws.append(output_row)

    out_wb.save(output_file)

    print("Arquivo gerado:", output_file)


# ============================================================
# MAIN
# ============================================================

# Directly call the processing function within Colab
process_workbook(
    str(INPUT_PATH),
    str(OUTPUT_PATH)
)

Arquivo gerado: porsche_database_sanitized.xlsx


## Prepare Data for Dashboard

Before creating the dashboard, we need to load the sanitized data into a pandas DataFrame and perform some final data preparation steps, such as converting column types and handling missing values.

In [ ]:
import pandas as pd

# Load the sanitized data
sanitized_df = pd.read_excel('porsche_database_sanitized.xlsx')

# Initial data cleaning and preparation for dashboard
sanitized_df['SaleDateSanitized'] = pd.to_datetime(sanitized_df['SaleDateSanitized'], errors='coerce', format='%Y-%m-%d')
sanitized_df['ModelYearSanitized'] = pd.to_numeric(sanitized_df['ModelYearSanitized'], errors='coerce').astype('Int64') # Use Int64 for nullable integer
sanitized_df['SalesPriceSanitized'] = pd.to_numeric(sanitized_df['SalesPriceSanitized'], errors='coerce')
sanitized_df['VehicleMileageSanitized'] = pd.to_numeric(sanitized_df['VehicleMileageSanitized'], errors='coerce').astype('Int64')

# Drop rows with INVALID or NaN in critical columns for dashboard analysis
sanitized_df.dropna(subset=['SaleDateSanitized', 'PorscheModelSanitized', 'ModelYearSanitized', 'CitySanitized', 'PayMethodSanitized', 'SalesPriceSanitized'], inplace=True)

# Display data information and head to verify
print("Data Types after preparation:")
display(sanitized_df.info())
print("\nFirst 5 rows of the prepared data:")
display(sanitized_df.head())

Data Types after preparation:
<class 'pandas.core.frame.DataFrame'>
Index: 47 entries, 2 to 98
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   sale_id                  47 non-null     int64         
 1   sale_date                47 non-null     object        
 2   SaleDateSanitized        47 non-null     datetime64[ns]
 3   customer_name            47 non-null     object        
 4   porsche_model            47 non-null     object        
 5   PorscheModelSanitized    47 non-null     object        
 6   model_year               47 non-null     object        
 7   ModelYearSanitized       47 non-null     Int64         
 8   sale_price               47 non-null     object        
 9   SalesPriceSanitized      47 non-null     float64       
 10  vehicle_mileage          47 non-null     object        
 11  VehicleMileageSanitized  47 non-null     Int64         
 12  payment_metho

None


First 5 rows of the prepared data:


,sale_id,sale_date,SaleDateSanitized,customer_name,porsche_model,PorscheModelSanitized,model_year,ModelYearSanitized,sale_price,SalesPriceSanitized,...,VehicleMileageSanitized,payment_method,PayMethodSanitized,city,CitySanitized,state,StateSanitized,salesperson,delivery_status,DeliveryStatusSanitized
2,8,2024-04-18 00:00:00,2024-04-18,Daniel-Jones,Cayenne Coupe,Cayenne Coupe,2023,2023,USD 112.750,112750.0,...,6400,Financing plan,Financiamento,austin,Austin,tx,TX,Brian Hall,in Transit,Em trânsito
4,10,2024-05-22 00:00:00,2024-05-22,Ethan Wilson,Taycan 4S,Taycan 4S,2024,2024,$121k,121.0,...,0,bank-transfer,Bank-Transfer,los angeles,Los Angeles,ca,CA,Thomas King,delivered!!!,Entregue
5,11,2024-08-06 00:00:00,2024-08-06,Ava Martinez,Panamera 4,Panamera 4,20-23,2023,"104,500 USD",104500.0,...,14500,Credit Card,Cartão de Crédito,Miami,Miami,FL,FL,LISA ray,cancelled,Cancelado
6,12,2024.07.11,2024-07-11,Noah Davis,911 Carrera S,911 Carrera S,2020,2020,"USD $96,300",96300.0,...,41000,lease,Arrendamento,new york,New York,New York,NY,Mark Evans,DELIVERD,Entregue
9,15,2024-09-02 00:00:00,2024-09-02,Mia Hernandez,Macan GTS,Macan GTS,2024,2024,"95,000 dollars",95000.0,...,3500,Financing,Financiamento,phoenix,Phoenix,arizona,AZ,Helen Brooks,IN TRANSIT,Em trânsito


In [43]:
import panel as pn
import altair as alt

pn.extension('tabulator', 'vega') # Ensure vega is enabled for Altair charts
pn.config.sizing_mode = "stretch_width" # Set a global sizing mode for responsiveness

# Define available options for filters
available_models = sorted(sanitized_df['PorscheModelSanitized'].unique().tolist())
available_years = sorted(sanitized_df['ModelYearSanitized'].dropna().unique().tolist())
available_cities = sorted(sanitized_df['CitySanitized'].unique().tolist())
available_payments = sorted(sanitized_df['PayMethodSanitized'].unique().tolist())

# Create filter widgets
model_selector = pn.widgets.MultiSelect(
    name='Porsche Model',
    options=available_models,
    value=available_models,
    size=8,
    width=200
)

year_slider = pn.widgets.IntSlider(
    name='Model Year',
    start=int(min(available_years)), end=int(max(available_years)),
    value=int(max(available_years)),
    step=1,
    width=200
)

city_selector = pn.widgets.MultiSelect(
    name='City',
    options=available_cities,
    value=available_cities,
    size=8,
    width=200
)

payment_selector = pn.widgets.MultiSelect(
    name='Payment Method',
    options=available_payments,
    value=available_payments,
    size=8,
    width=200
)

# Add DateRangeSlider
min_date = sanitized_df['SaleDateSanitized'].min().date()
max_date = sanitized_df['SaleDateSanitized'].max().date()
date_range_slider = pn.widgets.DateRangeSlider(
    name='Sale Date Range',
    start=min_date,
    end=max_date,
    value=(min_date, max_date),
    width=200
)

# Custom Porsche-inspired colors
PORSCHE_RED = '#CC0000'
PORSCHE_GREY = '#333333'
PORSCHE_LIGHT_GREY = '#666666'
PORSCHE_WHITE = '#FFFFFF'
PORSCHE_BLACK = '#000000'

# Make get_filtered_data a regular function (remove @pn.depends)
def get_filtered_data_func(models, year, cities, payments, date_range):
    start_date, end_date = date_range
    filtered_df = sanitized_df[
        sanitized_df['PorscheModelSanitized'].isin(models) &
        (sanitized_df['ModelYearSanitized'] <= year) &
        sanitized_df['CitySanitized'].isin(cities) &
        sanitized_df['PayMethodSanitized'].isin(payments) &
        (sanitized_df['SaleDateSanitized'].dt.date >= start_date) &
        (sanitized_df['SaleDateSanitized'].dt.date <= end_date)
    ].copy() # Use .copy() to avoid SettingWithCopyWarning
    return filtered_df

# Create a reactive object that holds the filtered DataFrame
filtered_df_reactive = pn.bind(get_filtered_data_func,
                               model_selector.param.value,
                               year_slider.param.value,
                               city_selector.param.value,
                               payment_selector.param.value,
                               date_range_slider.param.value)

# KPI 1: Principais modelos de carros vendidos por cidade
# Remove @pn.depends as we will use pn.bind explicitly below
def models_by_city_chart(df):
    if df.empty:
        return pn.pane.Markdown("No data available for the selected filters.")

    # Calculate top models per city
    top_models = df.groupby(['CitySanitized', 'PorscheModelSanitized']).size().reset_index(name='count')
    top_models = top_models.loc[top_models.groupby('CitySanitized')['count'].idxmax()]

    chart = alt.Chart(top_models).mark_bar().encode(
        x=alt.X('count', title='Number of Sales', axis=alt.Axis(grid=False, titleColor=PORSCHE_WHITE, labelColor=PORSCHE_WHITE)),
        y=alt.Y('PorscheModelSanitized', sort='-x', title='Porsche Model', axis=alt.Axis(titleColor=PORSCHE_WHITE, labelColor=PORSCHE_WHITE)),
        color=alt.value(PORSCHE_RED), # Use Porsche red for bars
        tooltip=['CitySanitized', 'PorscheModelSanitized', 'count']
    ).properties(
        title='Top Porsche Model Sold Per City',
        background=PORSCHE_BLACK
    ).configure_title(
        color=PORSCHE_WHITE,
        fontSize=16
    ).configure_axis(
        labelColor=PORSCHE_WHITE,
        titleColor=PORSCHE_WHITE,
        gridColor=PORSCHE_LIGHT_GREY
    ).configure_view(
        strokeWidth=0
    ).interactive()
    return chart

# KPI 2: Ano de modelo de carro que mais saiu em período
# Remove @pn.depends as we will use pn.bind explicitly below
def model_year_distribution_chart(df):
    if df.empty:
        return pn.pane.Markdown("No data available for the selected filters.")

    chart = alt.Chart(df).mark_bar().encode(
        x=alt.X('ModelYearSanitized:O', title='Model Year', axis=alt.Axis(grid=False, titleColor=PORSCHE_WHITE, labelColor=PORSCHE_WHITE)), # :O for ordinal to treat years as discrete categories
        y=alt.Y('count()', title='Number of Sales', axis=alt.Axis(titleColor=PORSCHE_WHITE, labelColor=PORSCHE_WHITE)),
        color=alt.value(PORSCHE_RED), # Use Porsche red for bars
        tooltip=['ModelYearSanitized', 'count()']
    ).properties(
        title='Model Year Distribution',
        background=PORSCHE_BLACK
    ).configure_title(
        color=PORSCHE_WHITE,
        fontSize=16
    ).configure_axis(
        labelColor=PORSCHE_WHITE,
        titleColor=PORSCHE_WHITE,
        gridColor=PORSCHE_LIGHT_GREY
    ).configure_view(
        strokeWidth=0
    ).interactive()
    return chart


# KPI 3: Popular car sales website based on data in each city (interpreted as most popular models per city with sales count)
# Remove @pn.depends as we will use pn.bind explicitly below
def popular_cars_per_city_table(df):
    if df.empty:
        return pn.pane.Markdown("No data available for the selected filters.")

    popular_cars = df.groupby(['CitySanitized', 'PorscheModelSanitized']).size().reset_index(name='Sales Count')
    # Sort by sales count within each city to identify popular models
    popular_cars_sorted = popular_cars.sort_values(by=['CitySanitized', 'Sales Count'], ascending=[True, False])

    # Display as a Tabulator table for detailed view
    return pn.widgets.Tabulator(popular_cars_sorted, layout='fit_data', show_index=False, width=800, height=300,
                                stylesheets=[':host {--tabulator-bg-color: #2E343A; --tabulator-border-color:#4D565E; --tabulator-row-border-color: #4D565E; --tabulator-header-bg-color: #1E2328; --tabulator-header-border-color:#4D565E; --tabulator-header-font-color: #FFFFFF; --tabulator-font-color: #FFFFFF; --tabulator-selected-bg-color: #4D565E;}'])


# KPI 4: Overall Daily Sales (Line Chart)
def overall_daily_sales_chart(df):
    if df.empty:
        return pn.pane.Markdown("No data available for the selected filters.")
    daily_sales = df.groupby('SaleDateSanitized')['SalesPriceSanitized'].sum().reset_index()
    chart = alt.Chart(daily_sales).mark_line(point=True).encode(
        x=alt.X('SaleDateSanitized', title='Sale Date', axis=alt.Axis(titleColor=PORSCHE_WHITE, labelColor=PORSCHE_WHITE, format='%Y-%m-%d')),
        y=alt.Y('SalesPriceSanitized', title='Total Sales Price', axis=alt.Axis(titleColor=PORSCHE_WHITE, labelColor=PORSCHE_WHITE)),
        tooltip=[alt.Tooltip('SaleDateSanitized', format='%Y-%m-%d'), 'SalesPriceSanitized']
    ).properties(
        title='Overall Daily Sales',
        background=PORSCHE_BLACK
    ).configure_title(
        color=PORSCHE_WHITE,
        fontSize=16
    ).configure_axis(
        labelColor=PORSCHE_WHITE,
        titleColor=PORSCHE_WHITE,
        gridColor=PORSCHE_LIGHT_GREY
    ).configure_view(
        strokeWidth=0
    ).interactive()
    return chart

# KPI 5: Sales by Payment Method (Bar Chart)
def sales_by_payment_chart(df):
    if df.empty:
        return pn.pane.Markdown("No data available for the selected filters.")
    payment_sales = df.groupby('PayMethodSanitized')['SalesPriceSanitized'].sum().reset_index()
    chart = alt.Chart(payment_sales).mark_bar().encode(
        x=alt.X('PayMethodSanitized', title='Payment Method', axis=alt.Axis(titleColor=PORSCHE_WHITE, labelColor=PORSCHE_WHITE)),
        y=alt.Y('SalesPriceSanitized', title='Total Sales Price', axis=alt.Axis(titleColor=PORSCHE_WHITE, labelColor=PORSCHE_WHITE)),
        color=alt.value(PORSCHE_RED),
        tooltip=['PayMethodSanitized', 'SalesPriceSanitized']
    ).properties(
        title='Sales by Payment Method',
        background=PORSCHE_BLACK
    ).configure_title(
        color=PORSCHE_WHITE,
        fontSize=16
    ).configure_axis(
        labelColor=PORSCHE_WHITE,
        titleColor=PORSCHE_WHITE,
        gridColor=PORSCHE_LIGHT_GREY
    ).configure_view(
        strokeWidth=0
    ).interactive()
    return chart

# KPI 6: Sales by State (Bar Chart)
def sales_by_state_chart(df):
    if df.empty:
        return pn.pane.Markdown("No data available for the selected filters.")
    state_sales = df.groupby('StateSanitized')['SalesPriceSanitized'].sum().reset_index()
    chart = alt.Chart(state_sales).mark_bar().encode(
        x=alt.X('StateSanitized', title='State', axis=alt.Axis(titleColor=PORSCHE_WHITE, labelColor=PORSCHE_WHITE)),
        y=alt.Y('SalesPriceSanitized', title='Total Sales Price', axis=alt.Axis(titleColor=PORSCHE_WHITE, labelColor=PORSCHE_WHITE)),
        color=alt.value(PORSCHE_RED),
        tooltip=['StateSanitized', 'SalesPriceSanitized']
    ).properties(
        title='Sales by State',
        background=PORSCHE_BLACK
    ).configure_title(
        color=PORSCHE_WHITE,
        fontSize=16
    ).configure_axis(
        labelColor=PORSCHE_WHITE,
        titleColor=PORSCHE_WHITE,
        gridColor=PORSCHE_LIGHT_GREY
    ).configure_view(
        strokeWidth=0
    ).interactive()
    return chart

# KPI 7: Price by Mileage (Scatter Plot)
def price_mileage_scatter_chart(df):
    if df.empty:
        return pn.pane.Markdown("No data available for the selected filters.")
    chart = alt.Chart(df).mark_circle().encode(
        x=alt.X('VehicleMileageSanitized', title='Vehicle Mileage', axis=alt.Axis(titleColor=PORSCHE_WHITE, labelColor=PORSCHE_WHITE)),
        y=alt.Y('SalesPriceSanitized', title='Sales Price', axis=alt.Axis(titleColor=PORSCHE_WHITE, labelColor=PORSCHE_WHITE)),
        color=alt.Color('PorscheModelSanitized', title='Porsche Model'),
        tooltip=['PorscheModelSanitized', 'VehicleMileageSanitized', 'SalesPriceSanitized']
    ).properties(
        title='Price by Mileage',
        background=PORSCHE_BLACK
    ).configure_title(
        color=PORSCHE_WHITE,
        fontSize=16
    ).configure_axis(
        labelColor=PORSCHE_WHITE,
        titleColor=PORSCHE_WHITE,
        gridColor=PORSCHE_LIGHT_GREY
    ).configure_legend(
        titleColor=PORSCHE_WHITE,
        labelColor=PORSCHE_WHITE
    ).configure_view(
        strokeWidth=0
    ).interactive()
    return chart


# Create reactive Panel objects for each KPI
models_by_city_chart_pane = pn.bind(models_by_city_chart, df=filtered_df_reactive)
model_year_distribution_chart_pane = pn.bind(model_year_distribution_chart, df=filtered_df_reactive)
popular_cars_per_city_table_pane = pn.bind(popular_cars_per_city_table, df=filtered_df_reactive)
overall_daily_sales_chart_pane = pn.bind(overall_daily_sales_chart, df=filtered_df_reactive)
sales_by_payment_chart_pane = pn.bind(sales_by_payment_chart, df=filtered_df_reactive)
sales_by_state_chart_pane = pn.bind(sales_by_state_chart, df=filtered_df_reactive)
price_mileage_scatter_chart_pane = pn.bind(price_mileage_scatter_chart, df=filtered_df_reactive)


# Dashboard Layout
dashboard_layout = pn.Column(
    pn.pane.HTML(
        "<div style='text-align: center; font-family: \"Porsche Next\", sans-serif; color: #FFFFFF; background-color: #1E2328; padding: 20px; font-size: 2.5em;'>" +
        "Porsche Sales Dashboard" +
        "</div>",
        sizing_mode='stretch_width'
    ),
    pn.Row(
        pn.Column(
            pn.pane.HTML(f"<h3 style='color: {PORSCHE_WHITE}; font-size: 1.5em; font-family: \"Porsche Next\", sans-serif;'>Filtros</h3>"),
            model_selector,
            year_slider,
            city_selector,
            payment_selector,
            date_range_slider, # Add the date range slider here
            height=700,
            sizing_mode='fixed',
            width=250,
            css_classes=['sidebar']
        ),
        pn.Column(
            pn.pane.HTML(f"<h3 style='color: {PORSCHE_WHITE}; font-size: 1.5em; font-family: \"Porsche Next\", sans-serif;'>Análises de Vendas</h3>"),
            pn.Tabs(
                ('Modelos por Cidade', models_by_city_chart_pane),
                ('Distribuição de Ano do Modelo', model_year_distribution_chart_pane),
                ('Carros Populares por Cidade', popular_cars_per_city_table_pane),
                ('Vendas Diárias', overall_daily_sales_chart_pane),
                ('Vendas por Método de Pagamento', sales_by_payment_chart_pane),
                ('Vendas por Estado', sales_by_state_chart_pane),
                ('Preço por Quilometragem', price_mileage_scatter_chart_pane), # New KPI
                active=0,
                stylesheets=[
                    ':host { --tabs-background: #1E2328; --tabs-color: #FFFFFF; --tabs-active-background: #CC0000; --tabs-active-color: #FFFFFF; }',
                    ':host .bk.bk-tabs-header .bk-tab { font-family: "Porsche Next", sans-serif; }'
                ]
            ),
            height=700,
            sizing_mode='stretch_width',
            css_classes=['main-content']
        ),
        sizing_mode='stretch_width'
    ),
    sizing_mode='stretch_width',
    css_classes=['porsche-dashboard'],
    # Add a custom CSS to embed Porsche fonts and refine design
    stylesheets=[
        """
        @import url('https://fonts.googleapis.com/css2?family=Porsche+Next:wght@300;400;700&display=swap');
        .porsche-dashboard {
            font-family: 'Porsche Next', sans-serif;
            background-color: #000000; /* Deep black */
            color: #FFFFFF;
        }
        .sidebar {
            background-color: #1E2328; /* Dark grey for sidebar */
            padding: 15px;
            border-radius: 8px;
            margin: 10px;
        }
        .main-content {
            background-color: #000000; /* Black for main content */
            padding: 15px;
            border-radius: 8px;
            margin: 10px;
        }
        .pn-widgets .bk-input {
            background-color: #333333 !important;
            color: #FFFFFF !important;
            border: 1px solid #666666 !important;
        }
        .pn-widgets .bk-input:focus {
            border-color: #CC0000 !important;
            box-shadow: 0 0 0 0.2rem rgba(204, 0, 0, 0.25) !important;
        }
        .pn-widgets label {
            color: #FFFFFF !important;
        }
        .bk-tab.bk-active {
            background-color: #CC0000 !important; /* Porsche Red for active tab */
            color: #FFFFFF !important;
        }
        .bk-tabs-header .bk-tab {
            background-color: #1E2328 !important;
            color: #FFFFFF !important;
            border-color: #333333 !important;
        }
        .bk-mark-rect.background {
            fill: #000000 !important;
        }
        """
    ]
)

# To display the dashboard (in Colab it renders below the cell)
dashboard_layout.servable()

Column(css_classes=['porsche-dashboard'], sizing_mode='stretch_width', stylesheets=["\n        @import url('h...])
    [0] HTML(str, sizing_mode='stretch_width')
    [1] Row(sizing_mode='stretch_width')
        [0] Column(css_classes=['sidebar'], height=700, sizing_mode='fixed', width=250)
            [0] HTML(str, sizing_mode='stretch_width')
            [1] MultiSelect(label='Porsche Model', name='Porsche Model', options=['718 Boxster', ...], size=8, value=['718 Boxster', ...], width=200)
            [2] IntSlider(end=2025, label='Model Year', name='Model Year', start=2020, value=2025, width=200)
            [3] MultiSelect(label='City', name='City', options=['Anchorage', ...], size=8, value=['Anchorage', ...], width=200)
            [4] MultiSelect(label='Payment Method', name='Payment Method', options=['Arrendamento', ...], size=8, value=['Arrendamento', ...], width=200)
            [5] DateRangeSlider(end=datetime.date(2027, ..., label='Sale Date Range', name='Sale Date Range', start=datetime.date(2024, ..., value=(datetime.date(2024, ..., value_end=datetime.date(2027, ..., value_start=datetime.date(2024, ..., width=200)
        [1] Column(css_classes=['main-content'], height=700, sizing_mode='stretch_width')
            [0] HTML(str, sizing_mode='stretch_width')
            [1] Tabs(sizing_mode='stretch_width', stylesheets=[':host { --tabs-backgroun...])
                [0] ParamFunction(function, _pane=Vega, defer_load=False, name='Modelos por Cidade', sizing_mode='stretch_width')
                [1] ParamFunction(function, _pane=Vega, defer_load=False, name='Distribuição d..., sizing_mode='stretch_width')
                [2] ParamFunction(function, _pane=Tabulator, defer_load=False, name='Carros Populares p..., sizing_mode='stretch_width')
                [3] ParamFunction(function, _pane=Vega, defer_load=False, name='Vendas Diárias', sizing_mode='stretch_width')
                [4] ParamFunction(function, _pane=Vega, defer_load=False, name='Vendas por Método d..., sizing_mode='stretch_width')
                [5] ParamFunction(function, _pane=Vega, defer_load=False, name='Vendas por Estado', sizing_mode='stretch_width')
                [6] ParamFunction(function, _pane=Vega, defer_load=False, name='Preço por Quilometragem', sizing_mode='stretch_width')

In [ ]:
# Removed redundant dashboard_layout.show() call